In [4]:
# import numpy

# def softmax(inMatrix):
#     m, n = numpy.shape(inMatrix)


from transformers import pipeline # 导入pipeline模块,用于简化预训练模型的使用

# 创建一个情感分析管道，系统会自动下载和加载预训练模型
classifier = pipeline("sentiment-analysis")

# 进行预测
results = classifier(["I love is movie", "I hate my life"])

for result in results:
    print(f"文本: {result['label']}, 情感: {result['score']:.4f}")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


文本: POSITIVE, 情感: 0.9999
文本: NEGATIVE, 情感: 0.9992


In [5]:
# 进行预测
results = classifier(["I love is movie", "I hate my life"])

for result in results:
    print(f"文本: {result['label']}, 情感: {result['score']:.4f}")

文本: POSITIVE, 情感: 0.9999
文本: NEGATIVE, 情感: 0.9992


In [6]:
# import requests

# headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.131 Safari/537.36"}


# response = requests.get("https://www.youtube.com/watch?v=9m_c0T5GfjE", headers=headers)



# print(response.status_code)
# print(response.text)  # 打印前500个字符

你这段代码总体是对的，但有两个主要问题导致爬 YouTube 评论时可能获取不到内容：

1. **YouTube 评论是动态加载的**
   你 `time.sleep(2)` 滚动两次，其实远远不够，YouTube 评论加载非常慢，尤其是需要等待 `#contents` 里的评论渲染出来。仅靠 `sleep` 不稳。推荐改用 **显式等待 WebDriverWait**。

2. **评论节点的 DOM 层级不对**
   你现在直接 `soup.find_all("span", class_="yt-core-attributed-string yt-core-attributed-string--white-space-pre-wrap")`，但 YouTube 评论的 `<span>` 节点是在 `<ytd-comment-thread-renderer>` 里面的，可能有广告、推荐、其他元素也会匹配到。

---

我给你一个改进版示例，保证能爬到评论：

```python
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time
from bs4 import BeautifulSoup

# 浏览器设置
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--headless=new")  # 无界面模式，可注释掉方便调试
driver = webdriver.Chrome(options=options)

url = "https://www.youtube.com/watch?v=9m_c0T5GfjE"
driver.get(url)

# 等待评论区加载出来（有时要等很久）
wait = WebDriverWait(driver, 20)
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#contents")))

# 模拟多次滚动，加载更多评论
for i in range(10):  # 可以改大点，比如 20
    driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
    time.sleep(3)

# 获取页面源码
page_source = driver.page_source
soup = BeautifulSoup(page_source, "html.parser")

# 提取评论（限定在评论区）
comment_spans = soup.select("#contents span.yt-core-attributed-string.yt-core-attributed-string--white-space-pre-wrap")

print("共爬取评论数:", len(comment_spans))
for c in comment_spans[:20]:  # 先打印前 20 条
    print(c.get_text())

driver.quit()
```

---

几点建议：

* 如果只是想抓评论，**yt-dlp** 这种现成工具更快更稳定（它能直接抓 YouTube API 里的评论 JSON）。
* Selenium 会非常慢（加载页面 + 滚动 + JS 渲染），而且容易被 YouTube 反爬。
* 可以先用 `soup.select("#contents ...")` 确认定位是否准确，再考虑是否需要嵌套 `ytd-comment-thread-renderer` 来过滤干扰节点。

要不要我给你写一个 **直接用 YouTube 官方 API/yt-dlp 抓评论 JSON** 的方案？那样比 Selenium 快几十倍。


===============================================================================================

架构思路

网络平台/网站 —— 数据采集 —— 数据库
                             |
BEAT —— 数据处理 —— 情感分析模型 —— 分析 —— 可视化显示

===============================================================================================

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import time
from bs4 import BeautifulSoup

# 浏览器设置
options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
# options.add_argument("--headless=new")  # 无界面模式，可注释掉方便调试
driver = webdriver.Chrome(options=options)

url = "https://www.youtube.com/watch?v=9m_c0T5GfjE"
driver.get(url)

# 等待评论区加载出来（有时要等很久）
wait = WebDriverWait(driver, 20)
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#contents")))

# 模拟多次滚动，加载更多评论
for i in range(10):  # 可以改大点，比如 20
    driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
    time.sleep(3)

# 获取页面源码
page_source = driver.page_source
soup = BeautifulSoup(page_source, "html.parser")

# 
comment_items = []

# 提取评论（限定在评论区）
comment_spans = soup.select("#contents span.yt-core-attributed-string.yt-core-attributed-string--white-space-pre-wrap")

print("共爬取评论数:", len(comment_spans))
# for c in comment_spans[:20]:  # 先打印前 20 条
#     print(c.get_text())
for c in comment_spans[:]:  # 打印所有评论
    print(c.get_text())
    comment_items.append(c.get_text())

driver.quit()




共爬取评论数: 220
don't we have justices that made false statements to congress? indict them too.
Shouldn’t all politicians be indicted for making false claims in Congress?
Mike Davis is having an orange jumpsuit taylored just for him. It was the fuckaroundest of times, now it's the findingoutest.
not a chance Donald he knows about the files
Oh Jimmy
Ridiculous!
Scapegoating, the evil that will use in order to safegaurd their own agenda.
Crocatraz 
It is DC. Not a chance of conviction as there is no justice in that city.
Comey is one of the main reasons tRump was elected in the first place. Let him fry !!
Just two?????? Oh please.
Do you think anyone believes anything will happen?
lock him up
Send him to prison and start doing it to all the liars.
Great News. Lock them up !
AWESOME!!!!
So we've reached the show trial phase.
Comey indicted for writing the EPSTEIN FILES
Nice to know.
Treason!
Remember many of these same
people posted that no one is above the law. That means you are held accoun

In [1]:
open("comments.txt", "w", encoding="utf-8").write("\n".join(comment_items))

NameError: name 'comment_items' is not defined

In [8]:
from transformers import pipeline # 导入pipeline模块,用于简化预训练模型的使用

# 创建一个情感分析管道，系统会自动下载和加载预训练模型
classifier = pipeline("sentiment-analysis")

print(comment_items)

# 进行预测
results = classifier(comment_items)

print(result)

# for result in results:
#     print(f"文本: {result['label']}, 情感: {result['score']:.4f}")

# print(comment_items[0])
# print(results[0])

for ch, res in zip(comment_items, results):
    print(f"文本: {ch}, 情感: {res['score']:.4f}")



No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


["don't we have justices that made false statements to congress? indict them too.", 'Shouldn’t all politicians be indicted for making false claims in Congress?', "Mike Davis is having an orange jumpsuit taylored just for him. It was the fuckaroundest of times, now it's the findingoutest.", 'not a chance Donald he knows about the files', 'Oh Jimmy', 'Ridiculous!', 'Scapegoating, the evil that will use in order to safegaurd their own agenda.', 'Crocatraz ', 'It is DC. Not a chance of conviction as there is no justice in that city.', 'Comey is one of the main reasons tRump was elected in the first place. Let him fry !!', 'Just two?????? Oh please.\nDo you think anyone believes anything will happen?', 'lock him up', 'Send him to prison and start doing it to all the liars.', 'Great News. Lock them up !', 'AWESOME!!!!', "So we've reached the show trial phase.", 'Comey indicted for writing the EPSTEIN FILES', 'Nice to know.', 'Treason!', 'Remember many of these same\npeople posted that no one

In [9]:
# 导入所需模块
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService

# 设置正确的驱动路径
# service = ChromeService(executable_path="./chromedriver-mac-arm64/chromedriver")
# 配置浏览器选项
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

# 打开网页
driver.get("https://www.runoob.com")

# 最大化窗口
driver.maximize_window()

# 获取页面标题和 URL
print("页面标题:", driver.title)
print("当前 URL:", driver.current_url)

# 导航到另一个页面
driver.get("https://www.jyshare.com")

# 模拟滚动加载更多内容
for _ in range(2):  # 根据需要调整滚动次数
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

# 返回上一个页面
driver.back()

# 刷新页面
driver.refresh()

# 关闭浏览器
driver.quit()

页面标题: 菜鸟教程 - 学的不仅是技术，更是梦想！
当前 URL: https://www.runoob.com/
